In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## ※ WITH句（Common Table Expression, CTE）

- `SELECT`の結果に一時的な名前を付け、後続のSQLでテーブルのように使用する方法。
- 作成したCTEは、そのSQL文の中だけで使用できる。
- 複雑なSQLを段階ごとに分けて書けるため、可読性が高くなる。
- 同じCTEを後続のSQLで複数回利用できる。

基本形：

```sql
WITH CTE名 AS (
    SELECT ...
)
SELECT *
FROM CTE名;
```

#### 1) 基本的なCTE

- `film`から必要なカラムだけを取得し、その結果に`movie`という一時的な名前を付けて使用する。

In [3]:
%%sql

WITH movie AS (
    SELECT
        title,
        rental_rate
    FROM film
)
SELECT *
FROM movie
LIMIT 5;

title,rental_rate
ACADEMY DINOSAUR,0.99
ACE GOLDFINGER,4.99
ADAPTATION HOLES,2.99
AFFAIR PREJUDICE,2.99
AFRICAN EGG,2.99


#### 2) CTE内で条件を指定

- CTE内の`WHERE`で、`rental_rate`が4以上の映画だけを抽出する。

In [5]:
%%sql

WITH expensive_movie AS (
    SELECT
        title,
        rental_rate
    FROM film
    WHERE rental_rate >= 4
)
SELECT *
FROM expensive_movie
LIMIT 5;

title,rental_rate
ACE GOLDFINGER,4.99
AIRPLANE SIERRA,4.99
AIRPORT POLLOCK,4.99
ALADDIN CALENDAR,4.99
ALI FOREVER,4.99


### 3) CTEの結果を並べ替え

- CTE内で120分以上の映画を抽出し、その結果を`length`の降順で並べ替える。

In [8]:
%%sql cte_order <<

WITH long_movie AS (
    SELECT
        title,
        length
    FROM film
    WHERE length >= 120
)
SELECT *
FROM long_movie
ORDER BY length DESC


[📊 全件の結果を見る](../../sql_study/results/7.15/cte_order.csv)

### 4. CTE内で値を加工

- `CONCAT()`で俳優の姓名を結合し、CTEの中で新しい`full_name`を作成する。

In [11]:
%%sql

WITH actor_name AS (
    SELECT
        actor_id,
        CONCAT(first_name, ' ', last_name) AS full_name
    FROM actor
)
SELECT *
FROM actor_name
ORDER BY full_name
LIMIT 5;

actor_id,full_name
71,ADAM GRANT
132,ADAM HOPPER
165,AL GARLAND
173,ALAN DREYFUSS
146,ALBERT JOHANSSON


#### 5) CTEの結果を後続のSELECTで絞り込み

- CTEで`title`と`rating`を取得し、後続の`WHERE`で`PG`の映画だけを抽出する。

In [13]:
%%sql

WITH movie AS (
    SELECT
        title,
        rating
    FROM film
)
SELECT *
FROM movie
WHERE rating = 'PG'
LIMIT 5;

title,rating
ACADEMY DINOSAUR,PG
AGENT TRUMAN,PG
ALASKA PHANTOM,PG
ALI FOREVER,PG
AMADEUS HOLY,PG
